# 4-1. 금융·공공 규제와 폐쇄망 LLM 아키텍처

⏱ **소요시간**: 30분

## 학습 목표

- 개인정보보호법 / 신용정보법 / 전자금융거래법 핵심 조항을 LLM 서비스 관점에서 이해한다.
- 금융감독원 "금융분야 AI 활용 안내서"의 권고사항이 LLM 파이프라인의 어느 단계에 매핑되는지 파악한다.
- 망분리(인터넷망 / DMZ / 내부망 / 자료전송 구간) 구조 위에 LLM 서비스를 배치하는 기준을 익힌다.
- 클라우드 API 방식 vs 폐쇄망 On-prem LLM 방식의 설계 체크리스트를 비교한다.

## 핵심 키워드

`개인정보보호법` · `가명정보` · `비식별` · `신용정보법` · `전자금융거래법` · `망분리` · `DMZ` · `자료전송시스템` · `금융분야 AI 활용 안내서` · `설명가능성(XAI)` · `On-prem LLM`

> 본 노트북은 **법률 자문이 아닌 엔지니어링 체크리스트**이다. 실제 서비스 출시 전에는 준법감시인(Compliance)·정보보호 담당(CISO)·감독당국 유권해석을 반드시 거친다.

## 1. 개인정보보호법 (PIPA) 핵심

| 조항 | 개념 | LLM 서비스 적용 포인트 |
| --- | --- | --- |
| §2 1호 | **개인정보** 정의: 살아있는 개인을 알아볼 수 있는 정보 (결합 용이성 포함) | 프롬프트·RAG 컨텍스트에 고객 식별자 포함 여부 사전 점검 |
| §2 1호의2 | **가명정보**: 추가 정보 없이는 특정 개인 식별 불가 | 학습/평가 데이터셋을 가명처리 후 별도 보관소(key-separated)에 저장 |
| §28-2 | **가명정보 처리 특례**: 통계작성·과학적 연구·공익적 기록보존 목적 활용 가능 | 내부 모델 fine-tuning용 데이터 생성 시 근거 |
| §17 | **제3자 제공** 동의 | 외부 LLM API(클라우드) 호출이 제3자 제공인지 **위탁**인지 해석 쟁점 |
| §26 | **처리위탁**: 위탁자 관리·감독 의무 | 클라우드 LLM 사용 시 위수탁계약·SCC·감사권 명시 필요 |
| §29 | **안전조치 의무** (접근통제·암호화·접속기록 보관) | LLM 서비스 감사로그·사용자 인증·TLS/저장 암호화 설계 |
| §39-3 | 개인정보 **국외 이전** 고지·동의 | 미국/EU 리전 LLM API 호출 시 국외이전 이슈 |

### 비식별 조치 (개인정보보호위원회 가이드라인)
- 가명처리, 총계처리, 데이터 삭제, 데이터 범주화, 데이터 마스킹
- **k-익명성 / l-다양성 / t-근접성** 같은 정량 지표로 재식별 위험 평가
- LLM 관점: RAG에 넣기 전에 문서 전처리 파이프라인에서 고정 규칙(정규식 + NER) + 샘플링 검토

## 2. 신용정보법 (신정법) 핵심

- **§17 신용정보 처리위탁**: 금융회사가 신용정보를 제3자에게 처리 위탁하려면 금융위 신고 또는 업무 위탁 기준 충족 필요.
- **§19 신용정보 보호조치**: 기술적·물리적·관리적 보호조치 + 내부 관리계획 수립.
- **§20 신용정보 이용·제공**: 목적 외 이용·제공 제한, 기록 보존 3년.
- **§40 개인신용정보 전송요구권 (마이데이터)**: API 기반 전송, 스크래핑 금지.

> 👉 LLM 프롬프트/응답에 **개인신용정보**(연체·잔액·소득·신용점수 등)가 포함될 경우 전자적 침해사고 대비 로그 보관·암호화 의무가 더 무거워진다.

## 3. 전자금융거래법 + 전자금융감독규정

### 망분리 의무 (전자금융감독규정)

| 조항 | 내용 | 엔지니어링 의미 |
| --- | --- | --- |
| §14 (정보처리시스템 구성) | 업무망·인터넷망 **물리적/논리적 분리** | LLM 추론 서버는 **내부망**(업무망)에 배치, 외부 API는 호출 금지 |
| §15 (해킹 등 방지대책) | 침입차단/탐지, 로그 5년 보관 등 | 모든 LLM 호출에 trace_id + 사용자ID + 입출력 로그 필요 |
| §31 (단말기 보안) | 업무 단말에서 외부망 접근 통제 | 개발자 PC에서 OpenAI 직접 호출 → 위반 소지 |
| §37-2 (클라우드 이용) | 중요도 평가 후 클라우드 이용 가능, **안전성 확보조치** 의무 | 클라우드 LLM 사용은 CSP 안전성 평가 + 금감원 보고 필요 |

### 자료전송시스템(자료연계)
- 외부망 ↔ 내부망 파일/데이터 이동은 반드시 **자료전송시스템**(일방향 게이트웨이, 악성코드 검사, 승인 워크플로)을 통과.
- LLM 모델 파일(GGUF, safetensors), pip wheel, 문서 데이터셋도 동일 절차.

## 4. 금융분야 AI 활용 안내서 (금융감독원) 요약

1. **데이터 거버넌스**: 학습 데이터 품질·편향·출처 관리, 개인정보/신용정보 적법근거 확보.
2. **모델 관리**: 모델 등록·버전관리·성능 모니터링·재학습 정책.
3. **설명가능성(XAI)**: 고객에게 중요한 결정(대출 거절 등)은 설명 근거 제공.
4. **공정성**: 민감 속성(성별·연령 등) 기반 차별 방지.
5. **안정성·보안**: 적대적 공격, 프롬프트 인젝션, 데이터 유출 대비.
6. **소비자 보호**: AI 이용 사실 고지, 인간 개입 채널 제공.

> LLM 관점에서는 ① RAG 출처 노출 ② 감사로그 ③ 가드레일(PII·프롬프트 인젝션) ④ 사람 핸드오프(Human-in-the-loop) 가 필수 구현 요소.

## 5. 망분리 구조와 LLM 서비스 배치 (Mermaid)

```mermaid
flowchart LR
    subgraph Internet["인터넷망"]
        U[고객 / 직원 BYOD]
        CS[(Cloud LLM API<br/>OpenAI / Anthropic)]
    end

    subgraph DMZ["DMZ (완충구역)"]
        WAF[WAF / Reverse Proxy]
        CDS[자료전송시스템<br/>일방향 게이트웨이]
    end

    subgraph Internal["내부 업무망 (폐쇄망)"]
        API[FastAPI / OpenWebUI]
        LLM[🔒 Ollama / vLLM<br/>on-prem GPU]
        VDB[(Vector DB<br/>Qdrant / pgvector)]
        KG[(Neo4j)]
        LOG[(감사로그 5년 보관)]
    end

    U -->|HTTPS| WAF --> API
    API --> LLM
    API --> VDB
    API --> KG
    API --> LOG
    CS -. 차단 / 예외승인 .- WAF
    CDS -->|모델·데이터 반입| LLM
```

**배치 원칙**

1. **추론 엔진(LLM)과 Vector DB**는 내부망에 둔다. 외부 네트워크 egress 차단.
2. **OpenWebUI/FastAPI**는 내부망에서 서빙, DMZ의 WAF를 통해서만 사용자에게 노출.
3. **모델·데이터셋 반입**은 자료전송시스템(일방향, 악성코드 스캔, 결재 승인)으로만 수행.
4. **감사로그**는 별도 저장소에 5년 이상 보관, 관리자 권한 분리(SoD).

## 6. 클라우드 API vs 폐쇄망 On-prem LLM 비교 체크리스트

| 항목 | ☁️ 클라우드 LLM API | 🔒 폐쇄망 On-prem LLM |
| --- | --- | --- |
| 개인정보 제3자 제공 / 국외 이전 | 해외 리전 시 §39-3 고지·동의, SCC 필요 | 내부망 내 처리 → 국외이전 이슈 없음 |
| 망분리 (전자금융감독규정 §14) | 외부망 egress 필요 → 위반 소지, 예외승인 필요 | 부합 |
| 클라우드 이용 (§37-2) | 중요도 평가·안전성 확보조치·금감원 보고 | 해당 없음 |
| 감사로그 5년 보관 | 벤더 SLA에 의존 + 자체 프록시 로깅 | 자체 보관 (통제 가능) |
| 모델 업데이트 | 즉시 반영, 재평가 필요 | 반입 절차 → 속도 느림, 변경 이력 명확 |
| 모델 품질 | 최신 frontier 모델 | qwen2.5, llama3 등 오픈모델, 자체 튜닝 가능 |
| 비용 | 토큰 단가 (가변) | 초기 GPU 투자 + 운영비 (고정) |
| 데이터 유출 위험 | API 호출 시 외부 전송 | 내부망 국한 |
| 설명가능성 (XAI) | 제한된 로그 | 전체 체인(프롬프트·리트리버·로짓) 기록 가능 |
| 인증·권한 | API Key 관리 | SSO·RBAC·MFA 직접 통합 |
| 재해복구(BCP) | 벤더 리전 의존 | 자체 HA·DR 설계 필요 |

### 실무 의사결정 가이드

- **민감등급 High (개인신용정보·내부문서)** → 폐쇄망 On-prem 필수.
- **민감등급 Low (공개 자료 요약·마케팅 카피)** → 클라우드 API 허용, 단 프록시 감사.
- **하이브리드**: 질의는 내부 sLLM, 고난도만 클라우드로 오프로드 + 사전 마스킹 (Day2 S5에서 시나리오 소개).

In [ ]:
# 현재 노트북의 provider 상태를 확인
import sys
sys.path.insert(0, '../..')
from common import provider_badge

print(provider_badge())